In [27]:
# import sys
# 本地没这个模块
# sys.path.append("/hpc/home/ephdh/workspace/mammo_foundation/baseline/src/utils")

import ast
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix, accuracy_score

# from analysis_utils import get_subgroup_df, generate_roc_curve, thres_eval_metric, pred_eval_metric


In [28]:
meta_data = pd.read_csv("../data/meta_data/meta_data_final.csv", encoding="utf-8")
png_index = pd.read_csv("../data/meta_data/case_png_index.csv", encoding="utf-8")

print("meta_data:", meta_data.shape)
print("png_index:", png_index.shape)

meta_data: (900, 22)
png_index: (3851, 4)


In [29]:
meta_data.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,OriginalPatientName,DensityCategory_std,DensityCategory_num,...,LesionType_cat,Histology_category,Histology_uncertain_reason,Density_std,Density_coarse,BIRADS_main,BIRADS_coarse,BIRADS_binary,LesionType_coarse,GroundTruth
0,徐琴芳,51,C,肿块,3.0,乳腺病,TN,徐琴芳,C,3.0,...,Mass,Not applicable (TN/FP),NaN,C,High density (C/D),3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0
1,沈诗怡,23,C,肿块,3.0,纤维腺瘤伴腺病,TN,沈诗怡,C,3.0,...,Mass,Not applicable (TN/FP),NaN,C,High density (C/D),3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0
2,龙菲菲,39,C,肿块,3.0,纤维腺瘤,TN,龙菲菲,C,3.0,...,Mass,Not applicable (TN/FP),NaN,C,High density (C/D),3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0
3,张菊,42,C,肿块,3.0,乳腺病伴局部导管上皮增生,TN,张菊,C,3.0,...,Mass,Not applicable (TN/FP),NaN,C,High density (C/D),3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0
4,刘星星,33,D,肿块,3.0,纤维腺瘤伴导管上皮普通型增生,TN,刘星星,D,4.0,...,Mass,Not applicable (TN/FP),NaN,D,High density (C/D),3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0


In [30]:
clip_results = pd.read_csv(
    "../data/meta_data/clip_results.csv", 
    # encoding="utf-16"
    )

clip_results = clip_results.drop_duplicates().reset_index(drop=True)

In [31]:
clip_results.head()

,img_path,scores
0,/hpc/home/ephdh/workspace/suzhou_false_validat...,0.02116
1,/hpc/home/ephdh/workspace/suzhou_false_validat...,0.03424
2,/hpc/home/ephdh/workspace/suzhou_false_validat...,0.08080
3,/hpc/home/ephdh/workspace/suzhou_false_validat...,0.10376
4,/hpc/home/ephdh/workspace/suzhou_false_validat...,0.02267


In [ ]:
# clip_results[clip_results['img_path'].duplicated(keep=False)]

In [32]:
print(clip_results.img_path.nunique(), clip_results.shape)

3849 (3849, 2)


In [ ]:
# clip_results['img_path'][0]

In [ ]:
# # 缺少suzhou_test-image-fold1.csv
# cnn_results = pd.read_csv(
#     "/hpc/home/ephdh/workspace/mammo_foundation/analysis/suzhou_test/suzhou_test-image-fold1.csv", 
#     # encoding="utf-16"
#     )

In [ ]:
# cnn_results.head()

In [ ]:
# print(clip_results.shape, cnn_results.shape)

In [ ]:
# merged_results = pd.merge(clip_results, cnn_results, 
#                           left_on='img_path', 
#                           right_on='NameList', 
#                           how='inner')
# merged_results = merged_results.rename(columns={
#     'scores': 'clip_image_score',
#     'Scores': 'cnn_image_score'})

In [ ]:
# print(merged_results.shape, merged_results['img_path'].nunique())

In [ ]:
# merged_results.head()

In [33]:
score_data = pd.read_csv("../data/meta_data/meta_data_with_model_scores.csv", encoding="utf-16")

KEYS = ["PatientName", "Group"]
SCORE_COLS = [
    "clip_patient_score",
    "cnn_patient_score",
    "ensemble_patient_score",
]

assert len(meta_data) == 900
assert len(score_data) == 900
assert not meta_data[KEYS].duplicated().any()
assert not score_data[KEYS].duplicated().any()

meta_keys = set(map(tuple, meta_data[KEYS].values))
score_keys = set(map(tuple, score_data[KEYS].values))
assert meta_keys == score_keys, (
    f"病例不一致\n"
    f"仅meta有：{meta_keys - score_keys}\n"
    f"仅score有：{score_keys - meta_keys}"
)

meta_data = meta_data.merge(
    score_data[KEYS + SCORE_COLS],
    on=KEYS,
    how="left",
    validate="one_to_one"
)

assert meta_data[SCORE_COLS].notna().all().all()

expected_ensemble = (meta_data["clip_patient_score"] + meta_data["cnn_patient_score"]) / 2

assert np.allclose(
    meta_data["ensemble_patient_score"],
    expected_ensemble,
    rtol=1e-10,
    atol=1e-12
)

AI_THRESHOLD = 0.536

meta_data["ai_score"] = meta_data["ensemble_patient_score"].astype(float)
meta_data["ai_pos"] = (
    meta_data["ai_score"] >= AI_THRESHOLD
).astype(int)

print("病例数:", len(meta_data))
print("ensemble_patient_score 校验通过")
print("AI threshold:", AI_THRESHOLD)

meta_data[
    [
        "PatientName",
        "Group",
        "GroundTruth",
        "clip_patient_score",
        "cnn_patient_score",
        "ensemble_patient_score",
        "ai_pos",
    ]
].head()

病例数: 900
ensemble_patient_score 校验通过
AI threshold: 0.536


,PatientName,Group,GroundTruth,clip_patient_score,cnn_patient_score,ensemble_patient_score,ai_pos
0,徐琴芳,TN,0,0.4985,0.310297,0.404398,0
1,沈诗怡,TN,0,0.7020,0.017702,0.359851,0
2,龙菲菲,TN,0,0.2449,0.142954,0.193927,0
3,张菊,TN,0,0.1987,0.959498,0.579099,1
4,刘星星,TN,0,0.2607,0.074582,0.167641,0


In [34]:
meta_data.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,OriginalPatientName,DensityCategory_std,DensityCategory_num,...,BIRADS_main,BIRADS_coarse,BIRADS_binary,LesionType_coarse,GroundTruth,clip_patient_score,cnn_patient_score,ensemble_patient_score,ai_score,ai_pos
0,徐琴芳,51,C,肿块,3.0,乳腺病,TN,徐琴芳,C,3.0,...,3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0,0.4985,0.310297,0.404398,0.404398,0
1,沈诗怡,23,C,肿块,3.0,纤维腺瘤伴腺病,TN,沈诗怡,C,3.0,...,3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0,0.7020,0.017702,0.359851,0.359851,0
2,龙菲菲,39,C,肿块,3.0,纤维腺瘤,TN,龙菲菲,C,3.0,...,3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0,0.2449,0.142954,0.193927,0.193927,0
3,张菊,42,C,肿块,3.0,乳腺病伴局部导管上皮增生,TN,张菊,C,3.0,...,3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0,0.1987,0.959498,0.579099,0.579099,1
4,刘星星,33,D,肿块,3.0,纤维腺瘤伴导管上皮普通型增生,TN,刘星星,D,4.0,...,3.0,3 Probably benign,Negative (<4),Mass/Nodule dominant,0,0.2607,0.074582,0.167641,0.167641,0


In [ ]:
# 无需单独评估clip
# test_df = meta_data[meta_data['Group'].isin(['TN', 'TP'])].reset_index(drop=True)

# all_scores = test_df.clip_patient_score.to_numpy()
# all_labels = test_df.GroundTruth.to_numpy()

# fpr, tpr, thresholds, optimal_threshold, aucs = generate_roc_curve(y_true=all_labels, y_score=all_scores, 
#                                                              interpolation=False, drop_intermediate=False, 
#                                                              sensitivity_target=None, 
#                                                              specificity_target=None, 
#                                                              method='youden', 
#                                                              return_auc=True)
# print(aucs)
# thres_eval_metric(y_true=all_labels, y_score=all_scores, threshold=optimal_threshold)

In [ ]:
# 无需单独评估cnn
# test_df = meta_data[meta_data['Group'].isin(['TN', 'TP'])].reset_index(drop=True)

# all_scores = test_df.cnn_patient_score.to_numpy()
# all_labels = test_df.GroundTruth.to_numpy()

# fpr, tpr, thresholds, optimal_threshold, aucs = generate_roc_curve(y_true=all_labels, y_score=all_scores, 
#                                                              interpolation=False, drop_intermediate=False, 
#                                                              sensitivity_target=None, 
#                                                              specificity_target=None, 
#                                                              method='youden', 
#                                                              return_auc=True)
# print(aucs)
# thres_eval_metric(y_true=all_labels, y_score=all_scores, threshold=optimal_threshold)

In [35]:
# ensemble_patient_score作为评分指标
from sklearn.metrics import roc_auc_score, confusion_matrix

AI_THRESHOLD = 0.536
test_df = meta_data[meta_data["Group"].isin(["TN", "TP"])].reset_index(drop=True)

y_true = test_df["GroundTruth"].astype(int).to_numpy()
y_score = test_df["ensemble_patient_score"].astype(float).to_numpy()
y_pred = (y_score >= AI_THRESHOLD).astype(int)

auc = roc_auc_score(y_true, y_score)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print("Radiologist-correct stratum (TP + TN)")
print("n =", len(test_df))
print(f"AUC:         {auc:.4f}")
print(f"Sensitivity: {sensitivity:.4%} ({tp}/{tp + fn})")
print(f"Specificity: {specificity:.4%} ({tn}/{tn + fp})")
print(f"Threshold:   {AI_THRESHOLD}")

Radiologist-correct stratum (TP + TN)
n = 500
AUC:         0.8523
Sensitivity: 69.2000% (173/250)
Specificity: 87.2000% (218/250)
Threshold:   0.536


In [36]:
# The rank-average ensemble has been retired. The only ensemble scale used
# downstream is the direct patient-level arithmetic mean defined above.
expected_patient_ensemble = (
    meta_data['clip_patient_score'].astype(float)
    + meta_data['cnn_patient_score'].astype(float)
) / 2
assert np.allclose(
    meta_data['ensemble_patient_score'].astype(float),
    expected_patient_ensemble,
    equal_nan=True,
), 'ensemble_patient_score is not the arithmetic mean of CLIP and CNN patient scores'

In [37]:
df = meta_data.copy()

In [38]:
df[['clip_patient_score', 'cnn_patient_score', 'ensemble_patient_score']].head()

,clip_patient_score,cnn_patient_score,ensemble_patient_score
0,0.4985,0.310297,0.404398
1,0.7020,0.017702,0.359851
2,0.2449,0.142954,0.193927
3,0.1987,0.959498,0.579099
4,0.2607,0.074582,0.167641


In [39]:
df = meta_data.copy()

# Ground truth
df["cancer"] = df["GroundTruth"].astype(int)
group_truth = df["Group"].isin(["TP", "FN"]).astype(int)
assert (df["cancer"] == group_truth).all(), "GroundTruth 与 Group 不一致"

# Radiologist binary prediction
df["rad_pos"] = df["Group"].isin(["TP", "FP"]).astype(int)

# AI score
df["ai_score"] = df["ensemble_patient_score"].astype(float)

# operating threshold
AI_THRESHOLD = 0.536
df["ai_pos"] = (df["ai_score"] >= AI_THRESHOLD).astype(int)

print("GroundTruth / Group consistency: passed")
print("AI threshold:", AI_THRESHOLD)

GroundTruth / Group consistency: passed
AI threshold: 0.536


In [40]:
def derive_outcomes(df):
    df = df.copy()
    y_true = df["cancer"].values
    rad = df["rad_pos"].values
    ai  = df["ai_pos"].values

    def outcome(pred, truth):
        if pred == 1 and truth == 1: return "TP"
        if pred == 0 and truth == 0: return "TN"
        if pred == 1 and truth == 0: return "FP"
        if pred == 0 and truth == 1: return "FN"

    df["rad_outcome"] = [outcome(r, y) for r, y in zip(rad, y_true)]
    df["ai_outcome"]  = [outcome(a, y) for a, y in zip(ai,  y_true)]
    return df


In [41]:
df = derive_outcomes(df)

assert (df["rad_outcome"] == df["Group"]).all()
print("Radiologist outcome consistency: passed")

Radiologist outcome consistency: passed


In [42]:
df.head()

,PatientName,PatientAge,DensityCategory,LesionType,BIRADSRisk,HistologicalSubtype,Group,OriginalPatientName,DensityCategory_std,DensityCategory_num,...,GroundTruth,clip_patient_score,cnn_patient_score,ensemble_patient_score,ai_score,ai_pos,cancer,rad_pos,rad_outcome,ai_outcome
0,徐琴芳,51,C,肿块,3.0,乳腺病,TN,徐琴芳,C,3.0,...,0,0.4985,0.310297,0.404398,0.404398,0,0,0,TN,TN
1,沈诗怡,23,C,肿块,3.0,纤维腺瘤伴腺病,TN,沈诗怡,C,3.0,...,0,0.7020,0.017702,0.359851,0.359851,0,0,0,TN,TN
2,龙菲菲,39,C,肿块,3.0,纤维腺瘤,TN,龙菲菲,C,3.0,...,0,0.2449,0.142954,0.193927,0.193927,0,0,0,TN,TN
3,张菊,42,C,肿块,3.0,乳腺病伴局部导管上皮增生,TN,张菊,C,3.0,...,0,0.1987,0.959498,0.579099,0.579099,1,0,0,TN,FP
4,刘星星,33,D,肿块,3.0,纤维腺瘤伴导管上皮普通型增生,TN,刘星星,D,4.0,...,0,0.2607,0.074582,0.167641,0.167641,0,0,0,TN,TN


In [43]:
assert "ensemble_score" not in df.columns
assert np.allclose(df["ai_score"], df["ensemble_patient_score"])
assert (df["ai_pos"] == (df["ensemble_patient_score"] >= 0.536).astype(int)).all()

df.to_csv(
    "../data/meta_data/meta_data_with_model_scores.csv",
    index=False,
    encoding="utf-16"
)

print("Saved:", df.shape)

Saved: (900, 31)
